# Polygon Labels Analysis

This notebook analyzes mapping fidelity labels from polygon-level labeling stored in GeoPackage files.

**Label Categories:**
- Large Overmapping
- Overmapping
- Accurate
- Undermapping
- Large Undermapping

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

locations = ['japan', 'porgera', 'lombok']

# Define colors for each category (matching the app colors)
color_map = {
    'large-overmapping': '#FF0000',
    'overmapping': '#FF8800',
    'accurate': '#00FF00',
    'undermapping': '#0088FF',
    'large-undermapping': '#0000FF'
}

## 1. Load Data from GeoPackage Files

In [ ]:
def load_geopackage_data(location):
    """
    Load GeoPackage data for a location.
    Follows naming convention: data/{location}/{location}.gpkg
    Alternative pattern: data/{location}/{location}_1.gpkg
    """
    gpkg_path = Path(f"../data/{location}/{location}.gpkg")
    
    # Try alternative naming pattern if primary doesn't exist
    if not gpkg_path.exists():
        gpkg_path = Path(f"../data/{location}/{location}_1.gpkg")
    
    if gpkg_path.exists():
        try:
            # Read GeoPackage
            gdf = gpd.read_file(gpkg_path)
            
            print(f"\n{'='*60}")
            print(f"Location: {location.upper()}")
            print(f"{'='*60}")
            print(f"File: {gpkg_path}")
            print(f"Total polygons: {len(gdf)}")
            
            # Check if 'label' column exists
            if 'label' not in gdf.columns:
                print(f"WARNING: 'label' column not found in {location}")
                print(f"Available columns: {gdf.columns.tolist()}")
                return None
            
            # Filter out rows with empty labels
            gdf_labeled = gdf[gdf['label'].notna() & (gdf['label'] != '')].copy()
            
            print(f"Labeled polygons: {len(gdf_labeled)}")
            print(f"Unlabeled polygons: {len(gdf) - len(gdf_labeled)}")
            print(f"Labeling coverage: {len(gdf_labeled)/len(gdf)*100:.1f}%")
            
            if len(gdf_labeled) > 0:
                print(f"\nColumns: {gdf_labeled.columns.tolist()}")
                print(f"\nFirst few labeled rows:")
                display(gdf_labeled.head())
                
                print(f"\nLabel distribution:")
                print(gdf_labeled['label'].value_counts())
            else:
                print(f"\nNo labeled polygons found for {location}")
                return None
            
            return gdf_labeled
            
        except Exception as e:
            print(f"Error loading {location}: {e}")
            return None
    else:
        print(f"\nGeoPackage file not found for {location}: {gpkg_path}")
        return None

# Load data for all locations
gdfs = {}
for location in locations:
    gdf = load_geopackage_data(location)
    if gdf is not None:
        gdfs[location] = gdf

## 2. Basic Statistics

In [ ]:
def basic_stats(location, gdf):
    """
    Calculate and display basic statistics for labeled polygons.
    """
    print(f"\n{'='*60}")
    print(f"STATISTICS FOR {location.upper()}")
    print(f"{'='*60}")
    
    # Overall statistics
    print("\nOVERALL STATISTICS")
    print("-" * 60)
    print(f"Total labeled polygons: {len(gdf)}")
    
    # Area statistics if available
    if 'area' in gdf.columns or 'Area_m' in gdf.columns:
        area_col = 'Area_m' if 'Area_m' in gdf.columns else 'area'
        total_area = gdf[area_col].sum()
        mean_area = gdf[area_col].mean()
        median_area = gdf[area_col].median()
        print(f"Total area (m²): {total_area:,.0f}")
        print(f"Mean polygon area (m²): {mean_area:,.0f}")
        print(f"Median polygon area (m²): {median_area:,.0f}")
    
    # Label category counts
    print("\nLABEL CATEGORY COUNTS")
    print("-" * 60)
    label_counts = gdf['label'].value_counts().sort_index()
    print(label_counts)
    
    # Percentages
    print("\nLABEL CATEGORY PERCENTAGES")
    print("-" * 60)
    label_pct = (gdf['label'].value_counts(normalize=True) * 100).sort_index()
    for label, pct in label_pct.items():
        print(f"{label:25s}: {pct:6.2f}%")
    
    # Area by label (if area column exists)
    if 'area' in gdf.columns or 'Area_m' in gdf.columns:
        area_col = 'Area_m' if 'Area_m' in gdf.columns else 'area'
        print("\nAREA BY LABEL CATEGORY (m²)")
        print("-" * 60)
        area_by_label = gdf.groupby('label')[area_col].sum().sort_index()
        for label, area in area_by_label.items():
            pct = area / total_area * 100
            print(f"{label:25s}: {area:12,.0f} m² ({pct:5.1f}%)")

# Run statistics for all locations
for location, gdf in gdfs.items():
    basic_stats(location, gdf)

## 3. Category Counts - Bar Chart

In [ ]:
def plot_category_counts(location, gdf):
    """
    Create bar chart showing label distribution.
    """
    # Get counts
    label_counts = gdf['label'].value_counts().sort_index()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Create bars with custom colors
    bars = ax.bar(range(len(label_counts)), label_counts.values,
                   color=[color_map.get(label, '#808080') for label in label_counts.index])
    
    # Customize plot
    ax.set_xticks(range(len(label_counts)))
    ax.set_xticklabels(label_counts.index, rotation=45, ha='right')
    ax.set_xlabel('Mapping Fidelity Category', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Polygons', fontsize=12, fontweight='bold')
    ax.set_title(f'Polygon Label Distribution - {location.capitalize()}',
                 fontsize=14, fontweight='bold', pad=20)
    
    # Add value labels on bars
    for i, (bar, count) in enumerate(zip(bars, label_counts.values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}\n({count/len(gdf)*100:.1f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Add grid
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    plt.savefig(f'../analysis/polygon_label_distribution_bar_{location}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Plot saved to: analysis/polygon_label_distribution_bar_{location}.png")

# Create plots for all locations
for location, gdf in gdfs.items():
    plot_category_counts(location, gdf)

## 4. Category Counts - Pie Chart

In [ ]:
def plot_category_pie(location, gdf):
    """
    Create pie chart showing label distribution.
    """
    # Get counts
    label_counts = gdf['label'].value_counts().sort_index()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create pie chart
    colors = [color_map.get(label, '#808080') for label in label_counts.index]
    wedges, texts, autotexts = ax.pie(label_counts.values,
                                        labels=label_counts.index,
                                        colors=colors,
                                        autopct='%1.1f%%',
                                        startangle=90,
                                        textprops={'fontsize': 11, 'fontweight': 'bold'})
    
    # Enhance text visibility
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(12)
    
    ax.set_title(f'Polygon Label Distribution - {location.capitalize()}',
                 fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(f'../analysis/polygon_label_distribution_pie_{location}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Plot saved to: analysis/polygon_label_distribution_pie_{location}.png")

# Create plots for all locations
for location, gdf in gdfs.items():
    plot_category_pie(location, gdf)

## 5. Area Analysis by Label Category

In [ ]:
def plot_area_by_label(location, gdf):
    """
    Analyze and visualize area distribution by label category.
    """
    # Check if area column exists
    if 'area' not in gdf.columns and 'Area_m' not in gdf.columns:
        print(f"No area column found for {location}")
        return
    
    area_col = 'Area_m' if 'Area_m' in gdf.columns else 'area'
    
    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Total area by category
    area_by_label = gdf.groupby('label')[area_col].sum().sort_index()
    colors = [color_map.get(label, '#808080') for label in area_by_label.index]
    
    bars = ax1.bar(range(len(area_by_label)), area_by_label.values, color=colors)
    ax1.set_xticks(range(len(area_by_label)))
    ax1.set_xticklabels(area_by_label.index, rotation=45, ha='right')
    ax1.set_xlabel('Label Category', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Total Area (m²)', fontsize=11, fontweight='bold')
    ax1.set_title('Total Area by Label Category', fontsize=12, fontweight='bold')
    ax1.yaxis.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, area in zip(bars, area_by_label.values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{area:,.0f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Plot 2: Box plot of polygon areas by category
    data_by_label = [gdf[gdf['label'] == label][area_col].values 
                     for label in sorted(gdf['label'].unique())]
    labels = sorted(gdf['label'].unique())
    
    bp = ax2.boxplot(data_by_label, labels=labels, patch_artist=True)
    
    # Color the box plots
    for patch, label in zip(bp['boxes'], labels):
        patch.set_facecolor(color_map.get(label, '#808080'))
        patch.set_alpha(0.6)
    
    ax2.set_xlabel('Label Category', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Polygon Area (m²)', fontsize=11, fontweight='bold')
    ax2.set_title('Area Distribution by Label Category', fontsize=12, fontweight='bold')
    ax2.yaxis.grid(True, alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.suptitle(f'Area Analysis - {location.capitalize()}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'../analysis/polygon_area_analysis_{location}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Plot saved to: analysis/polygon_area_analysis_{location}.png")
    
    # Print statistics
    print(f"\n{'='*60}")
    print(f"AREA STATISTICS BY LABEL - {location.upper()}")
    print(f"{'='*60}")
    for label in sorted(gdf['label'].unique()):
        label_data = gdf[gdf['label'] == label][area_col]
        print(f"\n{label}:")
        print(f"  Count: {len(label_data)}")
        print(f"  Total area: {label_data.sum():,.0f} m²")
        print(f"  Mean area: {label_data.mean():,.0f} m²")
        print(f"  Median area: {label_data.median():,.0f} m²")
        print(f"  Min area: {label_data.min():,.0f} m²")
        print(f"  Max area: {label_data.max():,.0f} m²")

# Create plots for all locations
for location, gdf in gdfs.items():
    plot_area_by_label(location, gdf)

## 6. Overmapping vs Undermapping Analysis

In [ ]:
def broad_category_analysis(location, gdf):
    """
    Analyze polygons grouped into broad categories.
    """
    # Group into broader categories
    def categorize_mapping(label):
        if 'overmapping' in label:
            return 'Overmapping'
        elif 'undermapping' in label:
            return 'Undermapping'
        else:
            return 'Accurate'
    
    gdf['broad_category'] = gdf['label'].apply(categorize_mapping)
    
    # Calculate statistics
    broad_counts = gdf['broad_category'].value_counts()
    
    print(f"\n{'='*60}")
    print(f"BROAD CATEGORY ANALYSIS - {location.upper()}")
    print(f"{'='*60}")
    print(broad_counts)
    print()
    
    # Calculate percentages
    for category, count in broad_counts.items():
        pct = count / len(gdf) * 100
        print(f"{category:15s}: {count:4d} polygons ({pct:5.1f}%)")
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    colors_broad = {'Overmapping': '#FF4400', 'Accurate': '#00FF00', 'Undermapping': '#0066FF'}
    bars = ax1.bar(range(len(broad_counts)), broad_counts.values,
                   color=[colors_broad.get(cat, '#808080') for cat in broad_counts.index])
    ax1.set_xticks(range(len(broad_counts)))
    ax1.set_xticklabels(broad_counts.index)
    ax1.set_ylabel('Number of Polygons', fontsize=11, fontweight='bold')
    ax1.set_title('Broad Category Distribution', fontsize=12, fontweight='bold')
    ax1.yaxis.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, count in zip(bars, broad_counts.values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}\n({count/len(gdf)*100:.1f}%)',
                ha='center', va='bottom', fontweight='bold')
    
    # Detailed breakdown
    detail_data = []
    detail_labels = []
    detail_colors = []
    
    for label in ['large-overmapping', 'overmapping', 'accurate', 
                  'undermapping', 'large-undermapping']:
        count = (gdf['label'] == label).sum()
        if count > 0:
            detail_data.append(count)
            detail_labels.append(label)
            detail_colors.append(color_map[label])
    
    ax2.pie(detail_data, labels=detail_labels, colors=detail_colors,
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax2.set_title('Detailed Category Breakdown', fontsize=12, fontweight='bold')
    
    plt.suptitle(f'Broad Category Analysis - {location.capitalize()}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'../analysis/polygon_broad_category_analysis_{location}.png',
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nPlot saved to: analysis/polygon_broad_category_analysis_{location}.png")

# Run analysis for all locations
for location, gdf in gdfs.items():
    broad_category_analysis(location, gdf)

## 7. Comparison Across Locations

In [ ]:
def compare_locations():
    """
    Compare label distributions across all locations.
    """
    if len(gdfs) == 0:
        print("No data available for comparison")
        return
    
    # Collect data for comparison
    comparison_data = []
    for location, gdf in gdfs.items():
        label_counts = gdf['label'].value_counts()
        for label, count in label_counts.items():
            comparison_data.append({
                'Location': location.capitalize(),
                'Label': label,
                'Count': count,
                'Percentage': count / len(gdf) * 100
            })
    
    comp_df = pd.DataFrame(comparison_data)
    
    # Create grouped bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Counts
    pivot_counts = comp_df.pivot(index='Label', columns='Location', values='Count').fillna(0)
    pivot_counts.plot(kind='bar', ax=ax1, width=0.8)
    ax1.set_xlabel('Label Category', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Number of Polygons', fontsize=11, fontweight='bold')
    ax1.set_title('Label Distribution by Location (Count)', fontsize=12, fontweight='bold')
    ax1.legend(title='Location', loc='upper right')
    ax1.yaxis.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Plot 2: Percentages
    pivot_pct = comp_df.pivot(index='Label', columns='Location', values='Percentage').fillna(0)
    pivot_pct.plot(kind='bar', ax=ax2, width=0.8)
    ax2.set_xlabel('Label Category', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Percentage (%)', fontsize=11, fontweight='bold')
    ax2.set_title('Label Distribution by Location (Percentage)', fontsize=12, fontweight='bold')
    ax2.legend(title='Location', loc='upper right')
    ax2.yaxis.grid(True, alpha=0.3)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.suptitle('Location Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../analysis/polygon_location_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Plot saved to: analysis/polygon_location_comparison.png")
    
    # Print comparison table
    print(f"\n{'='*80}")
    print("LOCATION COMPARISON TABLE")
    print(f"{'='*80}")
    print("\nCounts:")
    print(pivot_counts.to_string())
    print("\nPercentages:")
    print(pivot_pct.round(1).to_string())

# Run comparison
compare_locations()

## 8. Summary Statistics Table

In [ ]:
def create_summary_table():
    """
    Create summary statistics table for all locations.
    """
    if len(gdfs) == 0:
        print("No data available for summary")
        return
    
    # Collect summary data
    summary_data = []
    
    for location, gdf in gdfs.items():
        label_counts = gdf['label'].value_counts()
        
        for label in ['large-overmapping', 'overmapping', 'accurate', 
                      'undermapping', 'large-undermapping']:
            count = label_counts.get(label, 0)
            percentage = count / len(gdf) * 100 if len(gdf) > 0 else 0
            
            # Calculate area if available
            area_col = 'Area_m' if 'Area_m' in gdf.columns else ('area' if 'area' in gdf.columns else None)
            if area_col:
                total_area = gdf[gdf['label'] == label][area_col].sum()
            else:
                total_area = 0
            
            summary_data.append({
                'Location': location.capitalize(),
                'Label': label,
                'Count': int(count),
                'Percentage': f"{percentage:.1f}%",
                'Total Area (m²)': f"{total_area:,.0f}" if area_col else 'N/A'
            })
    
    summary_df = pd.DataFrame(summary_data)
    
    print(f"\n{'='*80}")
    print("SUMMARY STATISTICS TABLE")
    print(f"{'='*80}")
    print(summary_df.to_string(index=False))
    
    # Save to CSV
    summary_df.to_csv('../analysis/polygon_summary_statistics.csv', index=False)
    print(f"\nSummary saved to: analysis/polygon_summary_statistics.csv")
    
    # Overall totals
    print(f"\n{'='*80}")
    print("OVERALL TOTALS")
    print(f"{'='*80}")
    for location, gdf in gdfs.items():
        print(f"{location.capitalize():15s}: {len(gdf):5d} labeled polygons")
    print(f"{'TOTAL':15s}: {sum(len(gdf) for gdf in gdfs.values()):5d} labeled polygons")

# Create summary
create_summary_table()

## 9. Export Data for Further Analysis

In [ ]:
def export_enriched_data():
    """
    Export enriched datasets with broad categories.
    """
    def categorize_mapping(label):
        if 'overmapping' in label:
            return 'Overmapping'
        elif 'undermapping' in label:
            return 'Undermapping'
        else:
            return 'Accurate'
    
    for location, gdf in gdfs.items():
        # Add broad category
        gdf_export = gdf.copy()
        gdf_export['broad_category'] = gdf_export['label'].apply(categorize_mapping)
        
        # Export to GeoPackage
        output_path = f'../analysis/enriched_polygon_labels_{location}.gpkg'
        gdf_export.to_file(output_path, driver='GPKG')
        print(f"Exported: {output_path}")
        print(f"  Shape: {gdf_export.shape}")
        print(f"  Columns: {gdf_export.columns.tolist()}")
        print()

# Export data
export_enriched_data()

## 10. Key Findings Summary

In [ ]:
def print_key_findings():
    """
    Print key findings for each location.
    """
    def categorize_mapping(label):
        if 'overmapping' in label:
            return 'Overmapping'
        elif 'undermapping' in label:
            return 'Undermapping'
        else:
            return 'Accurate'
    
    for location, gdf in gdfs.items():
        print(f"\n{'='*80}")
        print(f" KEY FINDINGS - {location.upper()}")
        print(f"{'='*80}")
        
        # Most common label
        most_common = gdf['label'].mode()[0]
        most_common_count = (gdf['label'] == most_common).sum()
        print(f"\n1. Most Common Label: {most_common}")
        print(f"   ({most_common_count} polygons, {most_common_count/len(gdf)*100:.1f}%)")
        
        # Least common label
        least_common = gdf['label'].value_counts().idxmin()
        least_common_count = (gdf['label'] == least_common).sum()
        print(f"\n2. Least Common Label: {least_common}")
        print(f"   ({least_common_count} polygons, {least_common_count/len(gdf)*100:.1f}%)")
        
        # Broad categories
        gdf['broad_category'] = gdf['label'].apply(categorize_mapping)
        overmapping_count = (gdf['broad_category'] == 'Overmapping').sum()
        undermapping_count = (gdf['broad_category'] == 'Undermapping').sum()
        accurate_count = (gdf['broad_category'] == 'Accurate').sum()
        
        print(f"\n3. Broad Category Breakdown:")
        print(f"   Overmapping: {overmapping_count} polygons ({overmapping_count/len(gdf)*100:.1f}%)")
        print(f"   Undermapping: {undermapping_count} polygons ({undermapping_count/len(gdf)*100:.1f}%)")
        print(f"   Accurate: {accurate_count} polygons ({accurate_count/len(gdf)*100:.1f}%)")
        
        # Mapping quality assessment
        print(f"\n4. Overall Mapping Quality Assessment:")
        if accurate_count / len(gdf) > 0.6:
            print("   → HIGH QUALITY: Most polygons show accurate mapping")
        elif accurate_count / len(gdf) > 0.4:
            print("   → MODERATE QUALITY: Mixed mapping accuracy")
        else:
            print("   → NEEDS IMPROVEMENT: Significant mapping errors detected")
        
        if overmapping_count > undermapping_count * 1.5:
            print("   → SYSTEMATIC OVERMAPPING DETECTED")
        elif undermapping_count > overmapping_count * 1.5:
            print("   → SYSTEMATIC UNDERMAPPING DETECTED")
        else:
            print("   → BALANCED ERROR DISTRIBUTION")
        
        # Area statistics if available
        area_col = 'Area_m' if 'Area_m' in gdf.columns else ('area' if 'area' in gdf.columns else None)
        if area_col:
            total_area = gdf[area_col].sum()
            accurate_area = gdf[gdf['broad_category'] == 'Accurate'][area_col].sum()
            print(f"\n5. Area Statistics:")
            print(f"   Total labeled area: {total_area:,.0f} m²")
            print(f"   Accurate area: {accurate_area:,.0f} m² ({accurate_area/total_area*100:.1f}%)")

# Print findings
print_key_findings()